<a href="https://colab.research.google.com/github/sofiyaefimova302-png/Efimova-Sophya/blob/main/%22fine_tuning_hw_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [1]:
!pip install datasets transformers evaluate accelerate

# 1. Загружаю датасет ag_news
from datasets import load_dataset

dataset = load_dataset("ag_news")

# 2. Выбираю модель для дообучения
from transformers import AutoTokenizer

model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# 3. Токенизирую данные (max_length=128)
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Переименовываю целевую колонку для Trainer
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Задаю формат для PyTorch
tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# Создаю датасеты для обучения и тестирования
train_dataset = tokenized_datasets["train"]
test_dataset = tokenized_datasets["test"]

# 4. Загружаю модель
from transformers import AutoModelForSequenceClassification

num_labels = 4
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=num_labels)

# 5. Настраиваю TrainingArguments
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./ag_news_model",           # директория для сохранения модели
    learning_rate=2e-5,                     # learning_rate = 2e-5
    per_device_train_batch_size=16,         # per_device_train_batch_size = 16
    num_train_epochs=3,                     # num_train_epochs = 3
    eval_strategy="epoch",                  # eval_strategy = "epoch"
    save_strategy="epoch",                  # save_strategy = "epoch"
    load_best_model_at_end=True,            # load_best_model_at_end = True
    metric_for_best_model="accuracy",       # metric_for_best_model = "accuracy"
    report_to="none",                       # отключаю отчеты в сторонние сервисы
)

# 6. Определяю метрику accuracy
import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 7. Обучаю модель с помощью Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

# 8. Вывожу accuracy на тестовой выборке
eval_results = trainer.evaluate()
print(f"\nAccuracy на тестовой выборке: {eval_results['eval_accuracy']:.4f}")

# 9. Сохраняю модель в папку ./ag_news_model
model.save_pretrained("./ag_news_model")
tokenizer.save_pretrained("./ag_news_model")

# 10. Тестирую модель на трех новых новостях с использованием pipeline
from transformers import pipeline

# Загружаю сохраненную модель
classifier = pipeline("text-classification", model="./ag_news_model", tokenizer="./ag_news_model")

# Словарь для преобразования меток в названия категорий
label_names = ["World", "Sports", "Business", "Sci/Tech"]

# Три новые новости для тестирования (вписанные вручную)
new_news = [
    "The Russian team at the 2026 Winter Paralympic Games in Italy will be represented by six people.",
    "The story of Punch, the monkey from the Ichikawa Zoo in Japan, has touched the hearts of people all over the world.",
    "The LOFAR telescope has discovered 13.7 million previously unknown objects in the largest radio survey of the Universe."
]

# Тестирование
print("\n" + "="*60)
print("ТЕСТИРОВАНИЕ НА НОВЫХ НОВОСТЯХ")
print("="*60)

for i, news in enumerate(new_news):
    result = classifier(news)[0]
    label_id = int(result['label'].split('_')[-1])
    category = label_names[label_id]
    confidence = result['score']

    print(f"\nНовость {i+1}: {news}")
    print(f"Предсказанный класс: {category} (метка {label_id})")
    print(f"Уверенность модели: {confidence:.4f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

...

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Accuracy на тестовой выборке: 0.9488


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


ТЕСТИРОВАНИЕ НА НОВЫХ НОВОСТЯХ

Новость 1: The Russian team at the 2026 Winter Paralympic Games in Italy will be represented by six people.
Предсказанный класс: World (метка 0)
Уверенность модели: 0.6185

Новость 2: The story of Punch, the monkey from the Ichikawa Zoo in Japan, has touched the hearts of people all over the world.
Предсказанный класс: Sci/Tech (метка 3)
Уверенность модели: 0.6549

Новость 3: The LOFAR telescope has discovered 13.7 million previously unknown objects in the largest radio survey of the Universe.
Предсказанный класс: Sci/Tech (метка 3)
Уверенность модели: 0.9924
